# HERMES — LiteRT-LM CPU conversion trial

ใช้ Adapter เดิม ไม่ฝึกใหม่ ไม่ใช้ GPU และไม่สร้างทรัพยากร Google Cloud

**สถานะที่ยืนยันแล้ว:** Adapter ZIP ผ่าน SHA-256/CRC/manifest; โมเดล BF16 หลัง Merge ได้ 38/40 เคส ตัวแปลง V2 บน Work RAM 8 GiB หยุดด้วย OOM kill ในขั้น MLIR จึงยังไม่มีไฟล์ `.litertlm` ที่ผ่านทดสอบ

Notebook นี้เตรียมให้ลองบน Colab CPU ที่มี RAM มากกว่าเครื่อง Work แต่ **ยังไม่ได้รันครบใน Colab และไม่รับประกันว่า RAM ที่ได้รับจะเพียงพอ** ใช้ Runtime → CPU แล้ว Run all อนุญาต Drive เมื่อถาม ผลสำเร็จต้องมีการทดสอบ inference ผ่านก่อนข้อความ `CPU VERIFIED` การทดสอบมือถือยังแยกต่างหาก

ชุดทดสอบเป็นการจำแนกคำสั่ง 8 ประเภท 40 เคส ไม่ใช่การทดสอบผู้ช่วยทั่วไปหรือการเรียกใช้เครื่องมือ

## 1. เชื่อม Drive และตรวจเครื่อง

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime, timezone
import os,json,hashlib,zipfile,subprocess,shutil,sys,time
run_id=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
ROOT=Path('/content')/('hermes-litert-'+run_id)
ROOT.mkdir(exist_ok=False)
(ROOT/'logs').mkdir()
DRIVE_LAB=Path('/content/drive/MyDrive/HERMES_AI_Model_Lab')
assert DRIVE_LAB.is_dir(), 'HERMES_AI_Model_Lab not found in My Drive'
DEST=DRIVE_LAB/('HERMES_LiteRT_Trial_'+run_id)
DEST.mkdir(exist_ok=False)
os.environ['HERMES_RUN_ROOT']=str(ROOT)
def persist_logs():
    shutil.copytree(ROOT/'logs',DEST/'logs',dirs_exist_ok=True)
try:
    import psutil
    memory=psutil.virtual_memory()
    print('RAM total GiB:',round(memory.total/1024**3,2),'available:',round(memory.available/1024**3,2))
    (ROOT/'logs/resources.json').write_text(json.dumps({'ram_total_bytes':memory.total,'ram_available_bytes':memory.available,'disk_free_bytes':shutil.disk_usage('/content').free,'python':sys.version},indent=2))
finally:
    persist_logs()
assert shutil.disk_usage('/content').free>18*1024**3, 'Need 18 GiB free temporary disk for this experiment'
print('Results folder:',DEST)


## 2. ตรวจ Adapter ต้นฉบับและนำเฉพาะไฟล์ที่ต้องใช้

In [ ]:
ARCHIVE=DRIVE_LAB/'HERMES_20261012T162606Z_artifacts.zip'
EXPECTED_SHA='5ffd233321451f9e114e3c08eb6d9d93e25199ee29a88f470819a26b98c0d334'
def sha256(path):
    with Path(path).open('rb') as f:
        return hashlib.file_digest(f,'sha256').hexdigest()
assert ARCHIVE.is_file(), f'Adapter archive missing: {ARCHIVE}'
assert sha256(ARCHIVE)==EXPECTED_SHA, 'Adapter archive SHA-256 mismatch'
with zipfile.ZipFile(ARCHIVE) as z:
    assert z.testzip() is None, 'ZIP CRC failed'
    manifest=json.loads(z.read('manifest.json'))
    for item in manifest:
        b=z.read(item['path'])
        assert len(b)==item['bytes'] and hashlib.sha256(b).hexdigest()==item['sha256'], item['path']
    for name in z.namelist():
        if name.startswith('adapter/') or name in ['config.json','test.jsonl','validation.jsonl','train.jsonl']:
            target=ROOT/name
            assert ROOT in target.resolve().parents
            if not name.endswith('/'):
                target.parent.mkdir(parents=True,exist_ok=True)
                target.write_bytes(z.read(name))
config=json.loads((ROOT/'config.json').read_text())
assert config['model_id']=='Qwen/Qwen2.5-1.5B-Instruct'
assert config['base_revision']=='989aa7980e4cf806f80c7fef2b1adb7bc71aa306'
(ROOT/'logs/adapter-check.json').write_text(json.dumps({'archive_sha256':EXPECTED_SHA,'manifest_files_verified':len(manifest),'base_revision':config['base_revision']},indent=2))
persist_logs()
print('PASS: Adapter ZIP, CRC and all manifest hashes')


## 3. ติดตั้งเครื่องมือแปลงใน environment แยก

In [ ]:
ENV=ROOT/'venv'
subprocess.run([sys.executable,'-m','venv',str(ENV)],check=True)
PY=str(ENV/'bin/python')
def run_logged(args,logname,extra_env=None):
    env=os.environ.copy()
    env.update({'OMP_NUM_THREADS':'4','TOKENIZERS_PARALLELISM':'false','TF_CPP_MIN_LOG_LEVEL':'2'})
    if extra_env:env.update(extra_env)
    try:
        with (ROOT/'logs'/logname).open('w') as log:
            proc=subprocess.Popen(args,stdout=log,stderr=subprocess.STDOUT,env=env)
            while proc.poll() is None:
                time.sleep(20)
                lines=(ROOT/'logs'/logname).read_text(errors='replace').splitlines()
                if lines:print(logname,lines[-1][-240:],flush=True)
            if proc.returncode:
                print((ROOT/'logs'/logname).read_text(errors='replace')[-6000:])
                raise RuntimeError(f'{logname} failed with exit {proc.returncode}; logs saved to {DEST}')
    finally:
        persist_logs()
run_logged([PY,'-m','pip','install','torch==2.13.0','--index-url','https://download.pytorch.org/whl/cpu'],'install-torch.log')
run_logged([PY,'-m','pip','install','--pre','litert-torch-nightly==0.10.0.dev20261012','tf-nightly==2.23.0.dev20261012','peft==0.21.1','transformers==5.18.0','litert-lm==0.18.0'],'install-converter.log')
SOURCE=ROOT/'litert-torch'
SOURCE_COMMIT='0a81bf6f3243a6f7c846d05d62afeff4008f2542'
run_logged(['git','clone','https://github.com/google-ai-edge/litert-torch.git',str(SOURCE)],'source-fetch.log')
subprocess.run(['git','-C',str(SOURCE),'checkout','--detach',SOURCE_COMMIT],check=True)
assert subprocess.check_output(['git','-C',str(SOURCE),'rev-parse','HEAD'],text=True).strip()==SOURCE_COMMIT
EXPORT_ENV={'PYTHONPATH':str(SOURCE)}
run_logged([PY,'-c','from litert_torch._convert.converter_v2 import is_supported; assert is_supported(); print("V2 API READY")'],'converter-api-check.log',EXPORT_ENV)
(ROOT/'logs/source-commit.txt').write_text(SOURCE_COMMIT+'\n')
(ROOT/'logs/requirements-resolved.txt').write_text(subprocess.check_output([PY,'-m','pip','freeze'],text=True))
persist_logs()


## 4. ดาวน์โหลด Qwen รุ่นที่ตรงกับ Adapter

In [ ]:
download_script="""import os
from huggingface_hub import snapshot_download
snapshot_download(repo_id='Qwen/Qwen2.5-1.5B-Instruct',revision='989aa7980e4cf806f80c7fef2b1adb7bc71aa306',local_dir=os.path.join(os.environ['HERMES_RUN_ROOT'],'base'),allow_patterns=['*.json','*.safetensors','merges.txt','vocab.json'])
"""
run_logged([PY,'-c',download_script],'base-download.log')


## 5. Merge และทดสอบ 40 เคสเดิม

In [ ]:
(ROOT/'merge.py').write_text("import os,json,time,resource\nfrom pathlib import Path\nos.environ['HF_HUB_OFFLINE']='1'\nos.environ['TOKENIZERS_PARALLELISM']='false'\nimport torch\nfrom transformers import AutoModelForCausalLM,AutoTokenizer\nfrom peft import PeftModel\ntorch.set_num_threads(4)\nroot=Path(os.environ['HERMES_RUN_ROOT'])\nstarted=time.monotonic()\nbase=AutoModelForCausalLM.from_pretrained(root/'base',dtype=torch.bfloat16,device_map='cpu',low_cpu_mem_usage=True,trust_remote_code=False)\nprint('BASE LOADED',flush=True)\nmodel=PeftModel.from_pretrained(base,root/'adapter',is_trainable=False)\nmodel=model.merge_and_unload(safe_merge=True)\nprint('MERGE COMPLETE',flush=True)\nmodel.save_pretrained(root/'merged',safe_serialization=True,max_shard_size='1GB')\ntokenizer=AutoTokenizer.from_pretrained(root/'adapter',trust_remote_code=False)\ntokenizer.save_pretrained(root/'merged')\nresult={'stage':'merge','base_revision':'989aa7980e4cf806f80c7fef2b1adb7bc71aa306','dtype':'bfloat16','seconds':time.monotonic()-started,'peak_rss_kib':resource.getrusage(resource.RUSAGE_SELF).ru_maxrss}\n(root/'logs/merge-result.json').write_text(json.dumps(result,indent=2))\nprint(result,flush=True)\n",encoding='utf-8')

In [ ]:
(ROOT/'evaluate_merged.py').write_text("import os,json,time\nfrom pathlib import Path\nos.environ['HF_HUB_OFFLINE']='1'\nimport torch\nfrom transformers import AutoModelForCausalLM,AutoTokenizer\ntorch.set_num_threads(4)\nroot=Path(os.environ['HERMES_RUN_ROOT'])\nsystem='Classify the user request into one command. Reply with command name only. Commands: FIND (search work or records), INTAKE (receive a new request or brief), RETURN (send results back), STATUS (read progress), VERIFY (check correctness or evidence), REPORT (write a summary or report), DISCOVER (find suitable tools or capabilities), EXECUTE (perform an assigned authorized task). Do not execute actions; classify only.'\ntokenizer=AutoTokenizer.from_pretrained(root/'merged',trust_remote_code=False)\nmodel=AutoModelForCausalLM.from_pretrained(root/'merged',dtype=torch.bfloat16,low_cpu_mem_usage=True,trust_remote_code=False).eval()\npredictions=[]\nwith torch.inference_mode():\n for line in (root/'test.jsonl').read_text().splitlines():\n  row=json.loads(line)\n  text=tokenizer.apply_chat_template([{'role':'system','content':system},{'role':'user','content':row['query']}],tokenize=False,add_generation_prompt=True)\n  inputs=tokenizer(text,return_tensors='pt',add_special_tokens=False)\n  start=time.monotonic()\n  out=model.generate(**inputs,max_new_tokens=12,do_sample=False,pad_token_id=tokenizer.pad_token_id,eos_token_id=tokenizer.eos_token_id)\n  pred=tokenizer.decode(out[0,inputs['input_ids'].shape[1]:],skip_special_tokens=True).strip()\n  predictions.append({**row,'prediction':pred,'correct':pred==row['label'],'seconds':time.monotonic()-start})\n  print(len(predictions),pred,row['label'],flush=True)\n(root/'logs/merged-predictions.json').write_text(json.dumps(predictions,ensure_ascii=False,indent=2))\nresult={'correct':sum(r['correct'] for r in predictions),'total':len(predictions),'system_prompt':system,'production_accepted':False}\n(root/'logs/merged-evaluation.json').write_text(json.dumps(result,indent=2))\nprint(result,flush=True)\n",encoding='utf-8')

In [ ]:
(ROOT/'convert.py').write_text("import os,time,json,resource\nfrom pathlib import Path\nos.environ['HF_HUB_OFFLINE']='1'\nos.environ['TOKENIZERS_PARALLELISM']='false'\nos.environ['TF_CPP_MIN_LOG_LEVEL']='2'\nos.environ['OMP_NUM_THREADS']='4'\nimport torch\ntorch.set_num_threads(4)\nfrom litert_torch.generative.export_hf import export\nroot=Path(os.environ['HERMES_RUN_ROOT'])\nstarted=time.monotonic()\nconfig=dict(model=str(root/'merged'),output_dir=str(root/'litert-output'),prefill_lengths=[128],cache_length=512,quantization_recipe='dynamic_wi4b32_afp32',use_v2=True,delete_in_memory_params=True,model_dtype='bfloat16',bundle_litert_lm=True,trust_remote_code=False)\n(root/'logs/conversion-config.json').write_text(json.dumps(config,indent=2))\nprint('CONVERSION CONFIG',config,flush=True)\nexport.export(**config)\nfiles=list((root/'litert-output').glob('*.litertlm'))\nassert files, 'No LiteRT-LM output produced'\nresult={'stage':'converted_not_runtime_tested','outputs':[str(p) for p in files],'seconds':time.monotonic()-started,'peak_rss_kib':resource.getrusage(resource.RUSAGE_SELF).ru_maxrss}\n(root/'logs/conversion-result.json').write_text(json.dumps(result,indent=2))\nprint(result,flush=True)\n",encoding='utf-8')

In [ ]:
(ROOT/'evaluate_litert.py').write_text("import os\nimport json,time,hashlib,resource\nfrom pathlib import Path\nimport litert_lm\nroot=Path(os.environ['HERMES_RUN_ROOT'])\nmodel_path=next((root/'litert-output').glob('*.litertlm'))\nsystem=json.loads((root/'logs/merged-evaluation.json').read_text())['system_prompt']\nrows=[json.loads(line) for line in (root/'test.jsonl').read_text().splitlines()]\npredictions=[]\nstarted=time.monotonic()\nwith litert_lm.Engine(str(model_path),backend=litert_lm.Backend.CPU(thread_count=4),max_num_tokens=512) as engine:\n for row in rows:\n  start=time.monotonic()\n  with engine.create_conversation(system_message=system,sampler_config=litert_lm.SamplerConfig(top_k=1,temperature=0,seed=42),max_output_tokens=12,automatic_tool_calling=False) as conversation:\n   message=conversation.send_message(row['query'],max_output_tokens=12)\n   pred=str(message).strip()\n  result={**row,'prediction':pred,'correct':pred==row['label'],'seconds':time.monotonic()-start}\n  predictions.append(result)\n  print(len(predictions),json.dumps(result,ensure_ascii=False),flush=True)\n  (root/'logs/litert-predictions.json').write_text(json.dumps(predictions,ensure_ascii=False,indent=2))\ncorrect=sum(r['correct'] for r in predictions)\nsummary={'correct':correct,'total':len(rows),'accuracy':correct/len(rows),'passed':correct/len(rows)>=0.95,'runtime':'LiteRT-LM CPU','seconds':time.monotonic()-started,'size_bytes':model_path.stat().st_size,'sha256':hashlib.file_digest(model_path.open('rb'),'sha256').hexdigest(),'peak_rss_kib':resource.getrusage(resource.RUSAGE_SELF).ru_maxrss,'android_verified':False,'production_accepted':False}\n(root/'logs/litert-evaluation.json').write_text(json.dumps(summary,indent=2))\nprint(summary,flush=True)\n",encoding='utf-8')

In [ ]:
run_logged([PY,'-u',str(ROOT/'merge.py')],'merge.log')
run_logged([PY,'-u',str(ROOT/'evaluate_merged.py')],'merged-evaluation.log')
result=json.loads((ROOT/'logs/merged-evaluation.json').read_text())
print('Merged test:',result['correct'],'/',result['total'])
assert result['correct']/result['total']>=0.95, 'Merged model below test threshold'


## 6. แปลง LiteRT-LM แล้วเปิดรันทดสอบจริง

In [ ]:
run_logged([PY,'-u',str(ROOT/'convert.py')],'conversion.log',EXPORT_ENV)
files=list((ROOT/'litert-output').glob('*.litertlm'))
assert len(files)==1, f'Expected exactly one model output: {files}'
candidate=files[0]
run_logged([PY,'-u',str(ROOT/'evaluate_litert.py')],'litert-evaluation.log')
result=json.loads((ROOT/'logs/litert-evaluation.json').read_text())
print(json.dumps(result,indent=2))
assert result['passed'], 'LiteRT model generated but failed CPU evaluation; not accepted'


## 7. บันทึกโมเดลที่ผ่าน CPU และตรวจสำเนาใน Drive

In [ ]:
MODEL=DEST/'HERMES-Qwen2.5-1.5B-Q4B32-ctx512.litertlm'
assert not MODEL.exists(), 'Refusing overwrite'
partial=MODEL.with_suffix('.litertlm.partial')
assert not partial.exists()
shutil.copyfile(candidate,partial)
assert sha256(partial)==result['sha256'], 'Drive copy SHA-256 mismatch'
partial.rename(MODEL)
persist_logs()
verification={'model':MODEL.name,'size_bytes':MODEL.stat().st_size,'sha256':sha256(MODEL),'cpu_test':result,'android_verified':False,'production_accepted':False}
(DEST/'verification.json').write_text(json.dumps(verification,indent=2))
print('CPU VERIFIED:',MODEL)
print('SHA-256:',verification['sha256'])
print('Next: import in the Android app and test on the actual phone. Android is not yet verified.')
